# 03R. Random Forest 참조선 · 5 fold × 3 seed
**9week** · 계획서: [연구계획서.md](연구계획서.md) · 데이터·분할: [02](02_정제_분할_A0_흐름확인.ipynb)

**이 문서의 역할:** CNN이 아닌 전통적 방법의 점수를 **같은 데이터·같은 사람 분할·같은 판정 방식**으로 구한다. 이 연구의 비교 대상(A·B 계열)은 아니고, 두 가지 질문에 답하기 위한 참조선이다.
1. "CNN이 이 데이터에 더 적합한가?"를 말하려면, 같은 조건에서 다른 방법보다 나은지 확인해야 한다.
2. 다른 논문의 Random Forest 0.87은 평가 방식이 다를 수 있다. 사용자 독립 평가에서는 몇이 나오는지 직접 확인한다.

| 항목 | CNN (A0 등) | Random Forest (이 문서) |
|---|---|---|
| 입력 | 윈도우 원시 신호 (128, 3) | 윈도우마다 계산한 통계 특징 47개 |
| 특징 | 모델이 스스로 학습 | 사람이 정한 계산식 |
| 같은 것 | 정제, 분할, 윈도우(학습 stride 64 / 검증 128), 가중치 없는 학습, seed 3개, 지표 | |

특징은 원 연구(Kwapisz et al., 2010)가 사용한 평균·표준편차·평균 절대편차·합성 가속도 등을 참고한 **단순 통계 특징**이다. 원 연구의 특징을 정확히 재현한 것은 아니다.

## 1. 설정
- 데이터, 분할, 학습 설정은 **모든 모델 노트북에서 같다.** 바뀌는 것은 `MODEL_NAME`과 3절의 모델 코드뿐이다.
- 분할 ID는 02에서 확정한 값을 그대로 복사했다.

In [1]:
import os, re, sys, tarfile, time
import numpy as np
import pandas as pd
from collections import Counter

if 'google.colab' in sys.modules:                       # Colab: Google Drive 연결
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/가천대학교/2026-2/1. 기계학습프로그래밍'
else:                                                   # PC에서 실행할 때
    BASE = r'D:\가천대학교\26-2\1. 기계학습프로그래밍'
ARCHIVE = os.path.join(BASE, 'DATA', 'WISDM', 'WISDM_ar_latest.tar.gz')
RUN_DIR = os.path.join(BASE, '9week', 'runs')
os.makedirs(RUN_DIR, exist_ok=True)

MODEL_NAME = 'RF'
SEEDS = [0, 1, 2]
FOLDS = [0, 1, 2, 3, 4]

ACTIVITIES = ['Walking', 'Jogging', 'Upstairs', 'Downstairs', 'Sitting', 'Standing']
WIN, TRAIN_STRIDE, EVAL_STRIDE = 128, 64, 128
MAX_GAP_NS = 1_000_000_000

TEST_USERS = [6, 21, 24, 27, 29, 34]                    # 최종 평가 전까지 사용하지 않음
GROUPS = [[1, 7, 20, 22, 25, 36],
          [9, 16, 19, 26, 28, 33],
          [2, 3, 4, 14, 30, 31],
          [8, 10, 11, 12, 13, 15],
          [5, 17, 18, 23, 32, 35]]

## 2. 데이터 읽기와 정제 (02와 같은 규칙)
timestamp 0 삭제 → 완전 중복 삭제 → 같은 시각의 두 번째 값 삭제 → 사용자·행동 변경, 시간 역행, 1초 초과 공백에서 구간 끊기 → 128 이상 구간만 사용. 마지막 줄에 02와 같은 수(구간 424개, 윈도우 7,841개)가 나와야 한다.

In [2]:
with tarfile.open(ARCHIVE) as tar:
    raw = tar.extractfile('WISDM_ar_v1.1/WISDM_ar_v1.1_raw.txt').read().decode()

records = []
for token in re.split(r'[;\r\n]+', raw):
    token = token.strip()
    if token == '':
        continue
    fields = token.split(',')
    if len(fields) == 7 and fields[-1].strip() == '':
        fields = fields[:6]
    try:
        if len(fields) != 6:
            raise ValueError
        records.append((int(fields[0]), ACTIVITIES.index(fields[1]), int(fields[2]),
                        float(fields[3]), float(fields[4]), float(fields[5])))
    except ValueError:
        records.append(None)

seen, kept = set(), []
for r in records:
    if r is None:
        kept.append(None)
        continue
    if r[2] == 0 or r in seen:
        continue
    seen.add(r)
    last = kept[-1] if kept else None
    if last is not None and last[:2] == r[:2] and last[2] == r[2]:
        continue
    kept.append(r)

usable, start = [], 0
for i in range(1, len(kept) + 1):
    a = kept[i - 1]
    b = kept[i] if i < len(kept) else None
    if not (a is not None and b is not None and a[:2] == b[:2] and 0 < b[2] - a[2] <= MAX_GAP_NS):
        part = [r for r in kept[start:i] if r is not None]
        if len(part) >= WIN:
            usable.append((part[0][0], part[0][1], np.array([r[3:] for r in part], dtype=np.float32)))
        start = i

print('128 이상 구간:', len(usable), '/ 윈도우(stride 128):',
      sum((len(x) - WIN) // EVAL_STRIDE + 1 for u, a, x in usable))

128 이상 구간: 424 / 윈도우(stride 128): 7841


In [3]:
def make_windows(users, stride, mean, std):
    X, y, who = [], [], []
    for u, a, x in usable:
        if u not in users:
            continue
        x = (x - mean) / std
        for s in range(0, len(x) - WIN + 1, stride):
            X.append(x[s:s + WIN])
            y.append(a)
            who.append(u)
    return np.array(X, dtype=np.float32), np.array(y), np.array(who)

## 3. 윈도우 → 통계 특징 47개
축 x, y, z와 합성 가속도 크기 `m = √(x²+y²+z²)`(휴대폰 방향과 무관한 움직임 세기) 각각에 대해 11개, 축 사이 상관 3개를 계산한다.

| 특징 | 의미 |
|---|---|
| 평균, 중앙값 | 자세(중력 방향) |
| 표준편차, 평균 절대편차 | 흔들림의 크기 |
| 최솟값, 최댓값, 10·25·75·90 백분위 | 값의 분포 범위 |
| 평균 변화량 `mean(|x[t+1]−x[t]|)` | 신호가 얼마나 빠르게 변하는가 |
| 상관 xy, xz, yz | 축들이 함께 움직이는 방식 |

트리 모델은 값의 크기를 비교만 하므로 표준화가 필요 없다. 그래서 원시 값(mean 0, std 1로 '변환하지 않음')으로 윈도우를 만든다.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score

def features(X):
    m = np.sqrt((X ** 2).sum(axis=2, keepdims=True))          # (N,128,1) 합성 가속도 크기
    S = np.concatenate([X, m], axis=2)                        # (N,128,4): x, y, z, m
    f = [S.mean(1), S.std(1), np.abs(S - S.mean(1, keepdims=True)).mean(1),
         S.min(1), S.max(1), np.median(S, axis=1)]
    f += [np.percentile(S, q, axis=1) for q in (10, 25, 75, 90)]
    f += [np.abs(np.diff(S, axis=1)).mean(1)]
    xc = X - X.mean(1, keepdims=True)
    sd = X.std(1) + 1e-8
    for i, j in [(0, 1), (0, 2), (1, 2)]:
        f.append(((xc[:, :, i] * xc[:, :, j]).mean(1) / (sd[:, i] * sd[:, j]))[:, None])
    return np.concatenate(f, axis=1)                          # (N, 47)

no_scale = (np.zeros(3, dtype=np.float32), np.ones(3, dtype=np.float32))
print('특징 수:', features(make_windows(GROUPS[0], EVAL_STRIDE, *no_scale)[0][:2]).shape[1])

특징 수: 47


## 4. 5 fold × 3 seed 학습
CNN 노트북과 같은 반복 구조다. 트리 300개, 클래스 가중치 없음(CNN과 같은 조건). seed는 트리를 만들 때의 무작위성을 바꾼다.

In [5]:
rows, user_rows = [], []
for fold in FOLDS:
    val_users = GROUPS[fold]
    train_users = [u for i, g in enumerate(GROUPS) if i != fold for u in g]
    X_train, y_train, _ = make_windows(train_users, TRAIN_STRIDE, *no_scale)
    X_val, y_val, u_val = make_windows(val_users, EVAL_STRIDE, *no_scale)
    F_train, F_val = features(X_train), features(X_val)

    for seed in SEEDS:
        t0 = time.time()
        rf = RandomForestClassifier(n_estimators=300, n_jobs=4, random_state=seed)
        rf.fit(F_train, y_train)
        seconds = time.time() - t0
        pred = rf.predict(F_val)
        recalls = recall_score(y_val, pred, labels=range(6), average=None, zero_division=0)
        row = dict(model=MODEL_NAME, fold=fold, seed=seed,
                   accuracy=np.mean(pred == y_val), macro_f1=f1_score(y_val, pred, average='macro'),
                   train_sec=round(seconds, 1))
        row.update({f'recall_{n}': r for n, r in zip(ACTIVITIES, recalls)})
        rows.append(row)
        for u in val_users:
            m = u_val == u
            user_rows.append(dict(model=MODEL_NAME, fold=fold, seed=seed, user=u,
                                  windows=int(m.sum()), accuracy=np.mean(pred[m] == y_val[m])))
        print(f"fold {fold} seed {seed}: acc {row['accuracy']:.4f}  macroF1 {row['macro_f1']:.4f}  {seconds:4.0f}s")

runs = pd.DataFrame(rows)
users = pd.DataFrame(user_rows)
runs.to_csv(os.path.join(RUN_DIR, f'{MODEL_NAME}_runs.csv'), index=False)
users.to_csv(os.path.join(RUN_DIR, f'{MODEL_NAME}_users.csv'), index=False)

fold 0 seed 0: acc 0.8493  macroF1 0.8369     8s


fold 0 seed 1: acc 0.8515  macroF1 0.8400     7s


fold 0 seed 2: acc 0.8507  macroF1 0.8371     8s


fold 1 seed 0: acc 0.7133  macroF1 0.6976     8s


fold 1 seed 1: acc 0.7394  macroF1 0.7188     8s


fold 1 seed 2: acc 0.7124  macroF1 0.7114     7s


... (5 outputs left out)

fold 3 seed 2: acc 0.8277  macroF1 0.7856     8s


fold 4 seed 0: acc 0.8417  macroF1 0.8065     8s


fold 4 seed 1: acc 0.8332  macroF1 0.8015     8s


fold 4 seed 2: acc 0.8213  macroF1 0.7939     8s


## 5. 결과 요약
CNN 노트북과 같은 형식이다. CNN과의 비교는 A0 결과가 나온 뒤 6절에 추가한다.

In [6]:
per_fold = runs.groupby('fold')[['accuracy', 'macro_f1']].mean()
per_fold['macro_f1_seed_std'] = runs.groupby('fold')['macro_f1'].std()
print(per_fold.round(4), '\n')
print(f"{MODEL_NAME}: Macro F1 {per_fold['macro_f1'].mean():.4f} ± {per_fold['macro_f1'].std():.4f} (fold 간)"
      f" / Accuracy {per_fold['accuracy'].mean():.4f} ± {per_fold['accuracy'].std():.4f}")
print(f"Macro F1 ≥ 0.90인 fold: {(per_fold['macro_f1'] >= 0.90).sum()}/5")
print('\n행동별 recall (전체 평균)')
print(runs[[f'recall_{n}' for n in ACTIVITIES]].mean().round(3).to_string())
print('\n사람별 정확도 (seed 평균, 낮은 순 8명)')
print(users.groupby(['fold', 'user'])['accuracy'].mean().sort_values().head(8).round(3).to_string())

      accuracy  macro_f1  macro_f1_seed_std
fold                                       
0       0.8505    0.8380             0.0018
1       0.7217    0.7093             0.0108
2       0.7407    0.7077             0.0081
3       0.8279    0.7861             0.0033
4       0.8321    0.8006             0.0064 

RF: Macro F1 0.7683 ± 0.0578 (fold 간) / Accuracy 0.7946 ± 0.0589
Macro F1 ≥ 0.90인 fold: 0/5

행동별 recall (전체 평균)
recall_Walking       0.789
recall_Jogging       0.962
recall_Upstairs      0.516
recall_Downstairs    0.516
recall_Sitting       0.892
recall_Standing      0.975

사람별 정확도 (seed 평균, 낮은 순 8명)
fold  user
2     30      0.263
0     25      0.408
1     9       0.424
2     2       0.530
1     16      0.579
4     17      0.590
3     12      0.626
1     28      0.733



## 6. 결과 해석 (2026-11-09, PC CPU 실행, 15회 각 약 8초)

### 6-1. 점수: Macro F1 0.768 ± 0.058, Accuracy 0.795
- 0.90 이상인 fold는 **0/5**다.
- 다른 논문의 Random Forest 0.87보다 낮다. 그 논문의 평가 방식은 확인하지 않았다. 다만 무작위 분할이었다면 같은 사람이 학습과 평가에 섞이므로, 사용자 독립 평가인 여기서 더 낮게 나오는 것은 예상과 맞는다.
- 02의 A0 1회 결과(fold 0, Macro F1 0.912)와 같은 fold의 RF(0.838)를 비교하면 A0가 약 7%p 높다. 정식 비교는 03(A0 5 fold) 결과가 나온 뒤 같은 fold끼리 한다.

### 6-2. 약점: 계단 구분 (Upstairs·Downstairs recall 각 0.52)
- 통계 특징은 윈도우 안 값의 분포만 보고 **시간 순서를 버린다.** 계단 오르기·내리기·걷기는 평균과 흔들림의 크기가 비슷하고, 충격이 오는 순서와 모양이 다르다.
- CNN의 합성곱은 이 순서와 모양을 직접 본다. 그래서 CNN이 이 데이터에 적합하다는 가설의 근거가 된다. A0와의 정식 비교로 확인한다.

### 6-3. 점수를 흔드는 것은 seed가 아니라 사람 조합
| 흔들림의 종류 | 크기 (Macro F1) |
|---|---:|
| 같은 fold에서 seed만 바꿀 때 | 0.2~1.1%p |
| fold 사이 (사람 조합이 바뀔 때) | 5.8%p (0.708~0.838) |

사람 조합이 바뀔 때의 흔들림이 seed보다 **5~30배** 크다. 그래서 같은 fold끼리 짝지어 비교하는 판정 방식(계획서 5절)이 필요하다.

### 6-4. 방법과 상관없이 어려운 사람들
- 사용자 25의 정확도는 **0.408로 A0(02)와 똑같다.** 전혀 다른 두 방법이 같은 사람의 같은 부분을 틀린다는 뜻이다.
- 사용자 9, 16, 30, 2도 낮다.
- 이것은 구조의 문제가 아니라 **그 사람의 데이터가 다른 사람들과 다르다**는 신호다. 휴대폰 방향, 걸음걸이, 라벨 문제 등이 원인일 수 있다. CNN 결과에서도 같은 사람들이 낮게 나오는지 함께 본다.

### 다음
03(A0) 결과가 나오면 이 문서에 A0 vs RF의 같은 fold 비교(6절 판정 규칙)를 추가한다.
